<a href="https://colab.research.google.com/github/joelleweis/Letzebuergesch-Historiker/blob/main/Wikipedia_Workflow_Letzebuergesch_Historiker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time
import pandas as pd

LANG = "lb"
API_URL = f"https://{LANG}.wikipedia.org/w/api.php"
CATEGORY = "Kategorie:Lëtzebuergesch Historiker"

# Define a User-Agent header
HEADERS = {
    "User-Agent": "MyWikipediaBot/1.0 (your_email@example.com)" # Replace with your bot name and email
}

def get_category_members(category):
    members = []
    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": category,
        "cmlimit": "max",
        "cmnamespace": 0  # nur normale Artikelseiten
    }

    while True:
        r = requests.get(API_URL, params=params, headers=HEADERS)
        r.raise_for_status()
        data = r.json()

        members.extend([m["title"] for m in data["query"]["categorymembers"]])

        if "continue" in data:
            params.update(data["continue"])
        else:
            break

    return members


def get_article_text(title):
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "redirects": 1,
        "titles": title
    }

    r = requests.get(API_URL, params=params, headers=HEADERS)
    r.raise_for_status()
    data = r.json()

    page = next(iter(data["query"]["pages"].values()))
    return page.get("extract", "")


historiker = get_category_members(CATEGORY)
print(f"{len(historiker)} Historiker*innen gefunden")

rows = []

for name in historiker:
    print("Lade:", name)
    text = get_article_text(name)

    rows.append({
        "historiker": name,
        "text": text
    })

    time.sleep(0.5)

df = pd.DataFrame(rows)
df.to_csv("letzebuergesch_historiker_texte.csv", index=False, encoding="utf-8")

with open("letzebuergesch_historiker_texte.txt", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write("\n\n" + "="*80 + "\n")
        f.write(row["historiker"] + "\n")
        f.write("="*80 + "\n\n")
        f.write(row["text"])

print("Fertig: CSV und TXT erstellt.")

184 Historiker*innen gefunden
Lade: Lëscht vu lëtzebuergeschen Historiker
Lade: Jean-Louis André
Lade: Vincent Artuso
Lade: Alain Atten
Lade: Evamaria Bange
Lade: Charles Barthel
Lade: Théodore Bassing
Lade: Simone Beck
Lade: Romaine Berens
Lade: Isabelle Bernard-Lesceux
Lade: Jean Bertels
Lade: François Bertemes
Lade: Jean Bertholet
Lade: Henri Blackes
Lade: Philippe Henri Blasen
Lade: Lucien Blau
Lade: Martin Blum
Lade: Nicolas Bosseler
Lade: Roger Bour
Lade: Hubert Brasseur (1823)
Lade: Jean-Pierre Brimmeyr
Lade: Albert Calmes
Lade: Christian Calmes
Lade: Paul Cerf
Lade: Victor Conzemius
Lade: Pierre Decock
Lade: Arthur Diderrich
Lade: Émile Diderrich
Lade: Nicolas Didier
Lade: Paul Dostert
Lade: Fernand Emmel
Lade: Johannes Enen
Lade: Jean Engling
Lade: Jean Ensch
Lade: Al Estgen
Lade: Norbert Etringer
Lade: Pierre Even (Affekot)
Lade: Simone Feis
Lade: Paul Feltes
Lade: Joseph Flies
Lade: Jean-Pierre Franck
Lade: Frantz Funck-Brentano
Lade: Gilles Genot
Lade: Jean-Pierre Glaesener

In [ ]:
!pip install spacy
!python -m spacy download xx_ent_wiki_sm

In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("xx_ent_wiki_sm")

df = pd.read_csv("letzebuergesch_historiker_texte.csv")

edges = []

for _, row in df.iterrows():
    historiker = row["historiker"]
    text = str(row["text"])

    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE"]:
            edges.append({
                "historiker": historiker,
                "ort": ent.text,
                "entity_label": ent.label_
            })

edges_df = pd.DataFrame(edges)

# Duplikate entfernen
edges_df = edges_df.drop_duplicates()

edges_df.to_csv("historiker_orte_edges.csv", index=False, encoding="utf-8")

edges_df.head()

In [ ]:
edges_df.rename(columns={
    "historiker": "Source",
    "ort": "Target"
}).to_csv("gephi_edges_historiker_orte.csv", index=False, encoding="utf-8")

In [ ]:
edges_df.rename(columns={
    "historiker": "Source",
    "ort": "Target"
}).to_csv("gephi_edges_historiker_orte.csv", index=False, encoding="utf-8")

In [ ]:
from google.colab import files
files.download('letzebuergesch_historiker_texte.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>